# Visualization


A compiled `ETraceGraph` is useful only when its relationships can be inspected. This page uses `learner.report` and the graph fields to compare single-layer, multi-layer, and convolutional models.


## Single-Layer RNN

We start with the simplest case: a single recurrent layer followed by a linear readout.
The `ValinaRNNCell` contains one hidden state and one recurrent weight, and the `Linear`
readout has its own weight that feeds into the output.

In [1]:
import jax
import jax.numpy as jnp
import brainstate
import braintrace

In [2]:
class SingleLayerRNN(brainstate.nn.Module):
    def __init__(self, n_in, n_rec, n_out):
        super().__init__()
        self.rnn = braintrace.nn.ValinaRNNCell(n_in, n_rec)
        self.out = braintrace.nn.Linear(n_rec, n_out)

    def update(self, x):
        return self.out(self.rnn(x))


model = SingleLayerRNN(10, 32, 5)

# braintrace.compile initialises states, compiles the ETP graph, and returns a ready learner.
# We compile for a single unbatched sample (no batch_size), so the hidden state is (32,) and
# the recurrent op is the matrix-vector primitive etp_mv. verbose=2 prints full diagnostics.
learner = braintrace.compile(model, braintrace.D_RTRL, jnp.zeros(10), verbose=2)
learner.show_graph()

The hidden groups are:

   Group 0: [('rnn', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn', 'W', 'weight')  is associated with hidden group 0


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)


Compiler diagnostics (warnings / errors):

   [warning] relation_excluded_non_temporal: ETP primitive etp_mv (weight=('out', 'weight')) has no connected hidden states. It will be treated as a non-temporal parameter.



The hidden groups are:

   Group 0: [('rnn', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn', 'W', 'weight')  is associated with hidden group 0


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)





D:\BrainTrace\.venv\Lib\site-packages\braintrace\_compiler\hid_param_op.py:969: UserWarning: ETP primitive etp_mv (weight=('out', 'weight')) has no connected hidden states. It will be treated as a non-temporal parameter.
  _emit_no_relation_diag(


The output shows:

- **Hidden Group 0**: the hidden state of the `ValinaRNNCell` (path `('rnn', 'h')`)
- **Associated weight**: the recurrent weight inside the RNN cell (`('rnn', 'W', 'weight')`),
  eligibility-traced because it feeds the hidden state through an ETP primitive
- **Non-etrace weight**: the readout weight (`('out', 'weight')`). `braintrace.nn.Linear`
  *does* use an ETP primitive, but the readout's output is the network's final output and
  never flows back into a hidden state -- so the compiler reports it as a non-temporal
  parameter (still trained, just not through an eligibility trace). A matching
  `has no connected hidden states` warning is emitted at compile time.

This tells us that `D_RTRL` will maintain an eligibility trace for the recurrent weight,
tracking how it influences the hidden state over time.

## Using `learner.report` — the CompilationReport

Every learner returned by `braintrace.compile` carries a `CompilationReport` at `learner.report`. It aggregates the compiler's findings into a single inspectable object so you can programmatically query what was included, what was excluded, and why — without parsing log output.

Key members:
- `report.counts` — summary dict with keys `hidden_groups`, `etrace_weights`, `excluded_weights`, `warnings`, `errors`
- `report.etrace_weights` — list of weight paths that have eligibility traces
- `report.excluded_weights` — list of `(weight path, reason)` pairs excluded from online learning
- `report.dynamic_states` — list of non-hidden dynamic-state paths discovered by the compiler
- `report.diagnostics` — full list of `CompilationRecord` objects
- `report.show(level)` — print a human-readable summary (`1` = hidden groups + weight lists; `2` = also raw WARNING/ERROR diagnostics)

In [3]:
# report.show(level) prints a structured summary at the requested verbosity.
# level=1 shows hidden groups, etrace weights, and excluded weights.
learner.report.show(1)

# Programmatic access to the summary counts
print("Counts:", learner.report.counts)

# Which weights participate in online learning?
print("ETrace weights:", learner.report.etrace_weights)

# Which weights were excluded (e.g., non-temporal readouts)?
print("Excluded weights:", learner.report.excluded_weights)

The hidden groups are:

   Group 0: [('rnn', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn', 'W', 'weight')  is associated with hidden group 0


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)



Counts: {'hidden_groups': 1, 'etrace_weights': 2, 'excluded_weights': 1, 'warnings': 1, 'errors': 0}
ETrace weights: [(('rnn', 'W', 'weight'), [0]), (('rnn', 'W', 'weight'), [0])]
Excluded weights: [(('out', 'weight'), 'relation_excluded_non_temporal')]


## Understanding ETraceGraph

The compiled graph is an `ETraceGraph` named tuple with several key fields:

| Field | Type | Description |
|---|---|---|
| `module_info` | `ModuleInfo` | Jaxpr and state mappings extracted from the model |
| `hidden_groups` | `Sequence[HiddenGroup]` | Discovered hidden state groups |
| `hid_path_to_group` | `Dict[Path, HiddenGroup]` | Mapping from hidden state path to its group |
| `hidden_param_op_relations` | `Sequence[HiddenParamOpRelation]` | Weight-primitive-hidden connections |
| `hidden_perturb` | `HiddenPerturbation` or `None` | Perturbation structure for Jacobian computation |

Each `HiddenGroup` records a cluster of hidden states that are updated together in one
recurrent step. Each `HiddenParamOpRelation` records the connection between a weight
parameter and the hidden groups it feeds into through an ETP primitive.

Let's inspect these programmatically:

In [4]:
graph = learner.graph

print("=== Hidden Groups ===")
for g in graph.hidden_groups:
    print(f"  Group {g.index}: {g.num_state} state(s), shape {g.varshape}")
    print(f"    Paths: {g.hidden_paths}")

print("\n=== Weight-Primitive-Hidden Relations ===")
for i, r in enumerate(graph.hidden_param_op_relations):
    print(f"  Relation {i}:")
    # ``trainable_paths`` is a dict {trainable key -> owning ParamState path};
    # a single primitive may own several (e.g. {weight, bias}).
    print(f"    Trainable paths: {r.trainable_paths}")
    print(f"    Primitive: {r.primitive}")
    print(f"    Hidden groups: {[g.index for g in r.hidden_groups]}")

print("\n=== Perturbation ===")
print(f"  Has perturbation: {graph.hidden_perturb is not None}")

=== Hidden Groups ===
  Group 0: 1 state(s), shape (32,)
    Paths: [('rnn', 'h')]

=== Weight-Primitive-Hidden Relations ===
  Relation 0:
    Trainable paths: {'weight': ('rnn', 'W', 'weight'), 'bias': ('rnn', 'W', 'weight')}
    Primitive: etp_mv
    Hidden groups: [0]

=== Perturbation ===
  Has perturbation: True


The `HiddenGroup.num_state` property returns the total number of state variables in the group,
and `HiddenGroup.varshape` returns the shape of each state variable. The
`HiddenParamOpRelation.primitive` field identifies which ETP primitive connects the weight to
the hidden state -- here `etp_mv` (matrix-vector product); you will also see `etp_mm`
(matrix-matrix, e.g. under batching) and `etp_conv` (convolution) in other models.

## Two-Layer RNN

Stacking recurrent layers makes the compiled graph richer: there are more hidden states and
more weights, and the compiler must decide how to group the hidden states. Below we stack two
`GRUCell` layers and a linear readout, then read the structure straight off `show_graph()` --
rather than assuming one group per layer, we let the compiler tell us how it grouped things.

In [5]:
class TwoLayerRNN(brainstate.nn.Module):
    def __init__(self, n_in, n_rec, n_out):
        super().__init__()
        self.rnn1 = braintrace.nn.GRUCell(n_in, n_rec)
        self.rnn2 = braintrace.nn.GRUCell(n_rec, n_rec)
        self.out = braintrace.nn.Linear(n_rec, n_out)

    def update(self, x):
        h1 = self.rnn1(x)
        h2 = self.rnn2(h1)
        return self.out(h2)


model2 = TwoLayerRNN(10, 32, 5)

# Compile for a single unbatched sample (no batch_size).
learner2 = braintrace.compile(model2, braintrace.D_RTRL, jnp.zeros(10))
learner2.show_graph()

The hidden groups are:

   Group 0: [('rnn1', 'h')]
   Group 1: [('rnn2', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn1', 'Wz', 'weight')  is associated with hidden group 0
   Weight 1: ('rnn1', 'Wh', 'weight')  is associated with hidden group 0
   Weight 2: ('rnn2', 'Wz', 'weight')  is associated with hidden group 1
   Weight 3: ('rnn2', 'Wh', 'weight')  is associated with hidden group 1


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)
   Weight 1: ('rnn1', 'Wr', 'weight')  (excluded: relation_excluded_weight_to_weight)
   Weight 2: ('rnn2', 'Wr', 'weight')  (excluded: relation_excluded_weight_to_weight)





D:\BrainTrace\.venv\Lib\site-packages\braintrace\_compiler\hid_param_op.py:969: UserWarning: ETP primitive etp_mv (weight=('rnn1', 'Wr', 'weight')) reaches a hidden state only through another trainable ETP primitive (etp_mv). Per the non-parametric-tail invariant this weight is excluded from ETP; learn it by BPTT or rewire the architecture so its output flows directly into a hidden state.
  _emit_no_relation_diag(
D:\BrainTrace\.venv\Lib\site-packages\braintrace\_compiler\hid_param_op.py:969: UserWarning: ETP primitive etp_mv (weight=('rnn2', 'Wr', 'weight')) reaches a hidden state only through another trainable ETP primitive (etp_mv). Per the non-parametric-tail invariant this weight is excluded from ETP; learn it by BPTT or rewire the architecture so its output flows directly into a hidden state.
  _emit_no_relation_diag(


Notice that:

- The compiler reports **two hidden groups**, not one: Group 0 contains `('rnn1', 'h')`
  and Group 1 contains `('rnn2', 'h')`. Equal state widths do not imply that coupled layers
  are merged; the authoritative result is the grouping returned by `show_graph()`.
- Four parameter states are eligibility-traced: the update-gate (`Wz`) and candidate (`Wh`)
  weights of each layer. Each parameter reaches the hidden group of its own layer through a
  distinct `etp_mv` relation.
- The reset-gate weights (`Wr`) and the readout (`out`) are excluded for different reasons.
  Each `Wr` path reaches a hidden state only through another trainable ETP primitive, producing
  `relation_excluded_weight_to_weight`; `out` is non-temporal and produces
  `relation_excluded_non_temporal`.

Reading the graph and report together therefore identifies both the retained online-learning
paths and the exact reason for every exclusion.

In [6]:
# Inspect the two-layer graph programmatically via learner.graph and learner.report
graph2 = learner2.graph

print(f"Number of hidden groups: {len(graph2.hidden_groups)}")
print(f"Number of weight-hidden relations: {len(graph2.hidden_param_op_relations)}")

print("\nHidden groups:")
for g in graph2.hidden_groups:
    print(f"  Group {g.index}: {g.hidden_paths}")

print("\nRelations:")
for i, r in enumerate(graph2.hidden_param_op_relations):
    groups = [g.index for g in r.hidden_groups]
    # ``trainable_paths`` maps each trainable key to its owning ParamState path.
    print(f"  Weight {i}: {r.trainable_paths} -> hidden group(s) {groups}")

# Quick summary via the report
print("\nReport counts:", learner2.report.counts)

Number of hidden groups: 2
Number of weight-hidden relations: 4

Hidden groups:
  Group 0: [('rnn1', 'h')]
  Group 1: [('rnn2', 'h')]

Relations:
  Weight 0: {'weight': ('rnn1', 'Wz', 'weight'), 'bias': ('rnn1', 'Wz', 'weight')} -> hidden group(s) [0]
  Weight 1: {'weight': ('rnn1', 'Wh', 'weight'), 'bias': ('rnn1', 'Wh', 'weight')} -> hidden group(s) [0]
  Weight 2: {'weight': ('rnn2', 'Wz', 'weight'), 'bias': ('rnn2', 'Wz', 'weight')} -> hidden group(s) [1]
  Weight 3: {'weight': ('rnn2', 'Wh', 'weight'), 'bias': ('rnn2', 'Wh', 'weight')} -> hidden group(s) [1]

Report counts: {'hidden_groups': 2, 'etrace_weights': 8, 'excluded_weights': 3, 'warnings': 3, 'errors': 0}


## Convolutional Network

ETP primitives also support convolutional operations via `braintrace.nn.Conv2d`. When a
convolution operates **recurrently** on a hidden feature map -- reading the current map and
writing the result back -- the compiler discovers the connection between the convolution
kernel and the hidden state it updates. This demonstrates the generality of the graph
compilation: it works with any ETP primitive, not just matrix multiplication.

> **Note.** `braintrace.nn.Conv2d` takes a channel-last `in_size` of the form `(H, W, C)`
> (the spatial dimensions plus the number of input channels) and a separate `out_channels`.
> The kernel is only traced as an ETP parameter if its output reaches a hidden state
> *directly* -- an intervening shape-changing op (such as a `reshape` flattening the feature
> map before a `Linear`) breaks the connection, just like the slice in the *Limitations*
> tutorial.

In [7]:
class ConvRNN(brainstate.nn.Module):
    """A convolutional *recurrent* layer.

    The conv operates on its own hidden feature map ``self.h`` and writes the
    result back, so the conv kernel feeds a hidden state **directly** -- which is
    what lets the compiler trace it as an ETP parameter.
    """

    def __init__(self):
        super().__init__()
        # in_size is the channel-last spatial+channel shape (H, W, C).
        self.conv = braintrace.nn.Conv2d(in_size=(28, 28, 8), out_channels=8,
                                         kernel_size=3, padding='SAME')
        self.h = brainstate.HiddenState(jnp.zeros((28, 28, 8)))
        self.out = braintrace.nn.Linear(8 * 28 * 28, 10)

    def update(self, x):
        # x: (28, 28, 8) external drive in the same feature space as the hidden map.
        # The conv reads the recurrent hidden map and writes back into it, with no
        # shape-changing op in between, so the kernel -> hidden connection is traced.
        self.h.value = jax.nn.tanh(self.conv(self.h.value) + x)
        return self.out(self.h.value.reshape(-1))


model3 = ConvRNN()

learner3 = braintrace.compile(model3, braintrace.D_RTRL, jnp.zeros((28, 28, 8)), batch_size=1)
learner3.show_graph()

The hidden groups are:

   Group 0: [('h',)]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('conv', 'weight')  is associated with hidden group 0


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)





In this model:

- The `Conv2d` kernel **is** discovered as an ETP parameter (the `etp_conv` primitive),
  because the convolution operates on the recurrent hidden feature map `self.h` and writes
  the result back into it -- so the kernel feeds a hidden state directly, exactly like a
  recurrent matmul weight would.
- The single hidden group is the `(28, 28, 8)` convolutional feature map.
- The readout `Linear` is **non-temporal** -- its output does not flow back into a hidden
  state -- so the compiler excludes it from eligibility-trace tracking (it is still trained,
  just as a plain parameter learned through the loss, not an online trace).

The key requirement is that the conv output reach the hidden state *without* an intervening
shape-changing op (a `reshape`/slice) or another trainable ETP weight. Here it does, so the
compiler traces the conv kernel's influence on the recurrent hidden state -- showing that ETP
generalises beyond matrix multiplication to convolutional recurrence.

## Summary

Use `learner.report` for a reader-friendly overview and inspect `ETraceGraph` fields when validating exact relations. Visualization can reveal missing or unexpected paths, but it does not by itself establish gradient correctness.
